# 04 · Gobierno de datos aplicado · ORCA v1.0

Este notebook **demuestra** el Plan de Gobierno de Datos ([`documentacion/05_plan_gobierno_datos.md`](../documentacion/05_plan_gobierno_datos.md)) sobre el lote real.
Cada sección genera evidencia verificable y la guarda en `documentacion/gobierno/`:

| # | Artefacto obligatorio | Evidencia que produce este notebook |
|---|---|---|
| 1 | Catálogo / inventario de activos | `catalogo_activos.csv` (desde los manifiestos) |
| 2 | Diccionario de datos | `diccionario_mart.csv` (tipo, dominio, unidad, valores válidos y su cumplimiento) |
| 3 | Roles y responsabilidades (RACI) | RACI validado (exactamente una A por tarea) |
| 4 | Políticas y estándares | nomenclatura validada + retención simulada |
| 5 | Reglas de calidad | 6 dimensiones medidas + reglas del mart |
| 6 | Linaje y trazabilidad | `linaje.csv` con cadena de SHA-256 verificada |
| 7 | Clasificación y sensibilidad | `clasificacion_campos_DS-01.csv` (PII marcada y su tratamiento) |
| 8 | Control de acceso | `matriz_acceso.csv` + prueba de la función de acceso |

Marco: DAMA-DMBOK (dominios de gobierno, metadatos, calidad, seguridad, integración) y Ley 1581 de 2012.

In [1]:
from pathlib import Path
import json, sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks"))
pd.set_option("display.max_columns", 50, "display.width", 200, "display.max_colwidth", 80)
CURATED, MART, RAW = ROOT / "data" / "curated", ROOT / "data" / "mart", ROOT / "data" / "raw"
GOB = ROOT / "documentacion" / "gobierno"
GOB.mkdir(parents=True, exist_ok=True)
import gobierno, limpieza, calidad, riesgo, hashlib
def sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
man_cur = json.loads((CURATED / "DS-01_contratos_curated.manifest.json").read_text(encoding="utf-8"))
man_mart = json.loads((MART / "mart.manifest.json").read_text(encoding="utf-8"))
print("Lote:", man_cur["origen"], "· corte", man_cur["corte"])

Lote: data/raw/DS-01_secop2_contratos_20261105.parquet · corte 2026-11-05


## 1. Catálogo de activos

Se construye **desde los manifiestos** que deja el pipeline, no a mano: si mañana entra un lote nuevo, el catálogo se actualiza solo. Owner, Steward y custodio salen del RACI (§3 del Plan).

In [2]:
DUENOS = {"Owner": "Luis Ángel Seoanes Ovieso", "Steward": "Julian Amariles Taborda", "Custodio": "Katheryn Ramírez Chimá"}
META = {"DS-01": ("SECOP II – Contratos Electrónicos", "Colombia Compra Eficiente", "Diaria"),
        "DS-02": ("SECOP II – Procesos de Contratación", "Colombia Compra Eficiente", "Diaria"),
        "DS-03": ("DIVIPOLA – Códigos de municipios", "DANE", "Semestral")}
filas = []
for codigo in META:
    m = json.loads(riesgo.ultimo(RAW / f"{codigo}_*.manifest.json").read_text(encoding="utf-8"))
    nombre, pub, frec = META[codigo]
    filas.append({"codigo": codigo, "activo": nombre, "capa": "raw", "archivo": m["archivo"], "filas": m["filas"],
                  "columnas": len(m["columnas"]), "fuente": f"datos.gov.co/{m['dataset']}", "filtro": m["filtro_soql"],
                  "publicador": pub, "frecuencia": frec, "licencia": "CC BY-SA 4.0", "fecha_carga_utc": m["fin_utc"],
                  "clasificacion": m["clasificacion"], "sha256": m["sha256"][:16], **DUENOS})
filas.append({"codigo": "DS-01-C", "activo": "Contratos curados (PII seudonimizada)", "capa": "curated",
              "archivo": "DS-01_contratos_curated.parquet", "filas": man_cur["filas_curated"], "columnas": man_cur["columnas_curated"],
              "fuente": man_cur["origen"], "filtro": "", "publicador": "ORCA", "frecuencia": "Diaria", "licencia": "Uso interno",
              "fecha_carga_utc": man_cur["fin_utc"], "clasificacion": man_cur["clasificacion"], "sha256": man_cur["sha256_curated"][:16], **DUENOS})
for nombre, s in man_mart["salidas"].items():
    filas.append({"codigo": "MART-" + nombre.split("_", 1)[1][:3].upper(), "activo": nombre, "capa": "mart", "archivo": s["archivo"],
                  "filas": s["filas"], "columnas": s["columnas"], "fuente": "DS-01-C + DS-02", "filtro": "", "publicador": "ORCA",
                  "frecuencia": "Diaria", "licencia": "Reservado", "fecha_carga_utc": man_mart["fin_utc"],
                  "clasificacion": man_mart["clasificacion"], "sha256": s["sha256"][:16], **DUENOS})
catalogo = pd.DataFrame(filas)
catalogo.to_csv(GOB / "catalogo_activos.csv", index=False, encoding="utf-8-sig")
assert catalogo[["Owner", "Steward", "Custodio"]].notna().all().all(), "Todo activo debe tener dueño (principio 4)"
catalogo[["codigo", "activo", "capa", "filas", "columnas", "frecuencia", "clasificacion", "Owner", "Steward"]]

,codigo,activo,capa,filas,columnas,frecuencia,clasificacion,Owner,Steward
0,DS-01,SECOP II – Contratos Electrónicos,raw,294409,95,Diaria,Confidencial (contiene PII en claro),Luis Ángel Seoanes Ovieso,Julian Amariles Taborda
1,DS-02,SECOP II – Procesos de Contratación,raw,598422,57,Diaria,Confidencial (contiene PII en claro),Luis Ángel Seoanes Ovieso,Julian Amariles Taborda
2,DS-03,DIVIPOLA – Códigos de municipios,raw,1122,7,Semestral,Público,Luis Ángel Seoanes Ovieso,Julian Amariles Taborda
3,DS-01-C,Contratos curados (PII seudonimizada),curated,294409,76,Diaria,Interno (PII seudonimizada con HMAC-SHA256),Luis Ángel Seoanes Ovieso,Julian Amariles Taborda
4,MART-CON,mart_contratos_riesgo,mart,294409,39,Diaria,Confidencial · reservado al control fiscal (alertas no son hallazgos),Luis Ángel Seoanes Ovieso,Julian Amariles Taborda
5,MART-ENT,mart_entidades,mart,445,21,Diaria,Confidencial · reservado al control fiscal (alertas no son hallazgos),Luis Ángel Seoanes Ovieso,Julian Amariles Taborda
6,MART-PAR,mart_pares_entidad_proveedor,mart,42917,13,Diaria,Confidencial · reservado al control fiscal (alertas no son hallazgos),Luis Ángel Seoanes Ovieso,Julian Amariles Taborda
7,MART-MEN,mart_mensual,mart,35,6,Diaria,Confidencial · reservado al control fiscal (alertas no son hallazgos),Luis Ángel Seoanes Ovieso,Julian Amariles Taborda


## 2. Diccionario de datos del mart (con linaje por campo)

La definición viene de `gobierno.DICCIONARIO_MART`; el tipo, el dominio observado y el cumplimiento de los valores válidos se **miden** sobre el lote. El diccionario de la fuente (20 columnas de DS-01 con regla y resultado) está en `documentacion/Gobierno de datos/S2_Metadatos_EquipoORCA_Completo.xlsx` y las métricas de las 76 columnas de curated en `documentacion/calidad/metricas_por_columna_DS-01.csv`.

In [3]:
import re
c = pd.read_parquet(MART / "mart_contratos_riesgo.parquet")
VALIDA = {  # regla ejecutable de 'valores válidos' por columna
    "id_contrato": lambda s: s.str.match(r"^CO1\.PCCNTR\.\d+$") & ~s.duplicated(keep=False),
    "id_nit_entidad": lambda s: s.str.fullmatch(r"\d{9}"),
    "cod_municipio": lambda s: s.str.fullmatch(r"05\d{3}"),
    "val_contrato": lambda s: s >= 0,
    "cant_dias_adicionados": lambda s: s.between(0, 3650),
    "id_proveedor_seud": lambda s: s.str.fullmatch(r"[0-9a-f]{16}"),
    "num_score_riesgo": lambda s: s.between(0, 100),
    "niv_riesgo": lambda s: s.isin(["Alto", "Medio", "Bajo", "Sin señales"]),
    "url_proceso_secop": lambda s: s.str.startswith("https://community.secop.gov.co"),
    "cant_banderas": lambda s: s.between(0, 9),
}
filas = []
for col, desc, unidad, validos, sens, origen, transf in gobierno.DICCIONARIO_MART:
    s = c[col]
    if pd.api.types.is_bool_dtype(s) or str(s.dtype) == "boolean":
        dominio = f"True {int(s.sum()):,} · False {int((~s.astype(bool)).sum()):,}"
    elif pd.api.types.is_numeric_dtype(s) or pd.api.types.is_datetime64_any_dtype(s):
        dominio = f"{s.min()} … {s.max()}"
    else:
        dominio = f"{s.nunique():,} valores distintos"
    ok = VALIDA[col](s.dropna()) if col in VALIDA else None
    filas.append({"columna": col, "descripcion": desc, "tipo": str(s.dtype), "unidad": unidad, "valores_validos": validos,
                  "dominio_observado": dominio, "pct_nulos": round(100 * s.isna().mean(), 2),
                  "pct_cumple_validos": round(100 * ok.fillna(False).mean(), 3) if ok is not None else None,
                  "sensibilidad": sens, "origen": origen, "transformacion": transf})
diccionario = pd.DataFrame(filas)
diccionario.to_csv(GOB / "diccionario_mart.csv", index=False, encoding="utf-8-sig")
faltan = set(c.columns) - set(diccionario["columna"])
print("Columnas del mart sin definición en el diccionario:", faltan or "ninguna")
diccionario[["columna", "tipo", "unidad", "valores_validos", "dominio_observado", "pct_nulos", "pct_cumple_validos", "sensibilidad"]]

Columnas del mart sin definición en el diccionario: ninguna


,columna,tipo,unidad,valores_validos,dominio_observado,pct_nulos,pct_cumple_validos,sensibilidad
0,id_contrato,string,texto,patrón CO1.PCCNTR.<dígitos>; único,"294,409 valores distintos",0.00,100.000,Público
1,id_proceso_compra,string,texto,patrón CO1.BDOS.<dígitos>,"284,576 valores distintos",0.00,NaN,Público
2,nom_entidad,string,texto,no vacío,416 valores distintos,0.00,NaN,Público
3,id_nit_entidad,string,texto de 9 dígitos,^[0-9]{9}$,416 valores distintos,0.00,100.000,Público
4,cod_orden_entidad,string,categoría,Territorial · Nacional · Corporación Autónoma,3 valores distintos,0.00,NaN,Público
5,nom_municipio,string,categoría,125 municipios de Antioquia o 'Sin municipio',118 valores distintos,0.00,NaN,Público
6,cod_municipio,string,texto de 5 dígitos,^05[0-9]{3}$,117 valores distintos,6.97,100.000,Público
7,cod_modalidad,string,categoría,lista de SECOP II (13 valores),13 valores distintos,0.02,NaN,Público
8,cod_tipo_contrato,string,categoría,lista de SECOP II,20 valores distintos,0.00,NaN,Público
9,cod_causal_modalidad,string,categoría,lista de SECOP II,35 valores distintos,0.01,NaN,Público


## 3. Roles y RACI

La matriz del Plan (§3) se valida con la **regla de oro**: exactamente una A por tarea y al menos una R.

In [4]:
RACI = pd.read_excel(ROOT / "documentacion" / "plan_gobierno" / "ORCA-Plantilla-RACI-v1.0.xlsx", sheet_name=None)
print("Hojas de la plantilla:", list(RACI))
hoja = next(v for k, v in RACI.items() if "RACI" in k.upper())
hoja.head(25)

Hojas de la plantilla: ['Madurez', 'Roles', 'RACI']


,Parte 3 · Matriz RACI,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8
0,R = ejecuta · A = responde (solo uno por tarea) · C = consultado · I = infor...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,#,Bloque,Tarea de gobierno,Data Owner\nLuis Ángel Seoanes Ovieso,Data Steward\nIntegrante 2,Custodio técnico\nIntegrante 3,Consumidor / Analista\nIntegrante 3,Nº de A,Regla de oro
3,1,Estrategia y políticas,"Definir alcance, fuentes y preguntas de negocio (P1–P6)",A/R,C,C,C,NaN,NaN
4,2,Estrategia y políticas,Redactar y aprobar las políticas y estándares del Plan de Gobierno,A,R,C,C,NaN,NaN
5,3,Estrategia y políticas,Asignar Owner y Steward a cada activo de datos,A,R,C,I,NaN,NaN
6,4,Metadatos,Construir y mantener el catálogo de activos,I,A/R,R,C,NaN,NaN
7,5,Metadatos,"Elaborar el diccionario de datos (definición, tipo y dominio de cada campo)",I,A/R,C,C,NaN,NaN
8,6,Metadatos,"Registrar el linaje por lote (SoQL, filas, SHA-256, transformaciones)",I,C,A/R,I,NaN,NaN
9,7,Metadatos,Gestionar cambios de esquema de SECOP (columnas nuevas o renombradas),I,A,R,C,NaN,NaN


In [5]:
# La misma matriz del Plan de Gobierno §3, como datos, para validar la regla de oro
tareas = ["Alcance y preguntas P1–P6", "Políticas y estándares", "Asignar Owner/Steward", "Catálogo", "Diccionario", "Linaje por lote",
          "Cambios de esquema", "Reglas y umbrales de calidad", "Medir calidad por lote", "Reportar problemas de calidad",
          "Resolver incidentes de calidad", "Clasificar PII", "Seudonimización y sal", "Aprobar accesos", "Operar la ingesta",
          "Banderas y score de riesgo", "Publicar en el dashboard", "KPI de gobierno"]
m = ["A/R C C C", "A R C C", "A R C I", "I A/R R C", "I A/R C C", "I C A/R I", "I A R C", "A R C C", "I A R I", "I A I R",
     "I A R C", "A R C I", "A C R I", "A C R I", "I I A/R I", "A C C R", "A C I R", "I A/R C I"]
raci = pd.DataFrame([x.split() for x in m], index=tareas, columns=["Owner (Luis)", "Steward (Julian)", "Custodio (Katheryn)", "Analista (Jhon)"])
chequeo = pd.DataFrame({"cant_A": raci.apply(lambda f: sum("A" in v for v in f), axis=1),
                        "cant_R": raci.apply(lambda f: sum("R" in v for v in f), axis=1)})
assert (chequeo["cant_A"] == 1).all() and (chequeo["cant_R"] >= 1).all(), "La regla de oro no se cumple"
print("Regla de oro: OK en las", len(raci), "tareas · A por rol:", {c: int(raci[c].str.contains("A").sum()) for c in raci})
raci.to_csv(GOB / "raci_proyecto.csv", encoding="utf-8-sig")
raci

Regla de oro: OK en las 18 tareas · A por rol: {'Owner (Luis)': 9, 'Steward (Julian)': 7, 'Custodio (Katheryn)': 2, 'Analista (Jhon)': 0}


,Owner (Luis),Steward (Julian),Custodio (Katheryn),Analista (Jhon)
Alcance y preguntas P1–P6,A/R,C,C,C
Políticas y estándares,A,R,C,C
Asignar Owner/Steward,A,R,C,I
Catálogo,I,A/R,R,C
Diccionario,I,A/R,C,C
Linaje por lote,I,C,A/R,I
Cambios de esquema,I,A,R,C
Reglas y umbrales de calidad,A,R,C,C
Medir calidad por lote,I,A,R,I
Reportar problemas de calidad,I,A,I,R


## 4. Políticas y estándares aplicados

### 4.1 Nomenclatura
Toda columna del mart debe empezar por un prefijo del estándar (`id_`, `nom_`, `cod_`, `fec_`, `val_`, `cant_`, `pct_`, `ind_`, `url_`, `txt_`, `num_`, `niv_`) y estar en minúsculas `snake_case`. Todo archivo raw sigue `DS-NN_<dataset>_<AAAAMMDD>`.

In [6]:
tablas = {n: pd.read_parquet(MART / f"{n}.parquet") for n in man_mart["salidas"]}
nom = pd.DataFrame([{"tabla": n, "columnas": t.shape[1], "no_conformes": gobierno.columnas_no_conformes(t.columns)} for n, t in tablas.items()])
archivos = [p.name for p in RAW.iterdir() if p.name != ".gitkeep"]
print("Archivos raw fuera del estándar:", [a for a in archivos if not gobierno.PATRON_ARCHIVO_RAW.match(a)] or "ninguno")
display(pd.DataFrame(gobierno.PREFIJOS.items(), columns=["prefijo", "significado"]))
assert all(not x for x in nom["no_conformes"]), "Hay columnas fuera del estándar"
nom

Archivos raw fuera del estándar: ninguno


,prefijo,significado
0,id_,Identificador (llave o seudónimo)
1,nom_,Nombre o etiqueta legible
2,cod_,Código o categoría
3,fec_,Fecha
4,val_,Valor monetario en COP corrientes
5,cant_,Cantidad o conteo (entero)
6,pct_,Proporción entre 0 y 1
7,ind_,Indicador booleano (bandera)
8,url_,Enlace
9,txt_,Texto libre (enmascarado)


,tabla,columnas,no_conformes
0,mart_contratos_riesgo,39,[]
1,mart_entidades,21,[]
2,mart_pares_entidad_proveedor,13,[]
3,mart_mensual,6,[]


### 4.2 Retención (Ley 1581: principio de temporalidad)

In [7]:
display(pd.DataFrame([r.__dict__ for r in gobierno.RETENCION]))
purgar = gobierno.lotes_a_purgar()
print("Lotes raw que la política manda purgar hoy:", [p.name for p in purgar] or "ninguno",
      "\n(el orquestador los borra solo con --aplicar-retencion)")

,capa,lotes_a_conservar,dias_maximos,motivo
0,raw,3,90,Evidencia del lote para re-procesar; contiene PII: se guarda lo mínimo (Ley ...
1,curated,1,365,Se sobrescribe en cada corrida; el linaje queda en el manifiesto.
2,mart,1,365,Se sobrescribe en cada corrida; las alertas gestionadas se archivan en el re...
3,automatizacion,0,1825,"Registros de ejecución y alertas: 5 años, como soporte de la vigilancia fiscal."


Lotes raw que la política manda purgar hoy: ninguno 
(el orquestador los borra solo con --aplicar-retencion)


### 4.3 Uso aceptable
Las reglas de uso aceptable (finalidad, prohibición de reidentificar, alertas ≠ hallazgos) están en el Plan §6.3. Aquí se verifica la que es automatizable: **el mart no permite reidentificar** a una persona natural (no hay documento ni nombre cuando `ind_persona_natural` es verdadero).

In [8]:
pn = c[c["ind_persona_natural"]]
print("Personas naturales con NIT o nombre visibles en el mart:", int(pn["id_nit_proveedor"].notna().sum() + pn["nom_proveedor_juridico"].notna().sum()))

Personas naturales con NIT o nombre visibles en el mart: 0


## 5. Reglas de calidad medidas

### 5.1 Quality gate del lote (6 dimensiones, `calidad.py`)

In [9]:
rep = pd.read_csv(ROOT / "documentacion" / "calidad" / "reporte_calidad_DS-01.csv")
display(rep)
q = man_cur["calidad"]
print("Puntaje por dimensión:", q["puntaje_por_dimension"])
print(f"Puntaje del lote: {q['puntaje_lote']} % · umbral {q['umbral']} % → {q['decision']}")

,dimension,regla,descripcion,evaluados,cumplen,fallan,cumplimiento_pct
0,Completitud,C1,fecha_de_firma presente,294409,294409,0,100.00
1,Completitud,C2,valor_del_contrato presente,294409,294409,0,100.00
2,Completitud,C3,documento_proveedor presente,294409,293838,571,99.81
3,Completitud,C4,ciudad definida (no 'No Definido'),294409,274532,19877,93.25
4,Completitud,C5,modalidad_de_contratacion presente,294409,294347,62,99.98
5,Validez,V1,valor_del_contrato no negativo,294409,294409,0,100.00
6,Validez,V2,fecha_de_firma no es futura,294409,294409,0,100.00
7,Validez,V3,ciudad reconocida en DIVIPOLA,274532,273900,632,99.77
8,Exactitud,E1,valor_del_contrato mayor que cero,294409,291561,2848,99.03
9,Exactitud,E2,valor_del_contrato sin atípico (<= $100.000 millones),294409,294361,48,99.98


Puntaje por dimensión: {'Completitud': 98.61, 'Validez': 99.92, 'Exactitud': 99.5, 'Consistencia': 99.94, 'Unicidad': 100.0, 'Oportunidad': 100.0}
Puntaje del lote: 99.66 % · umbral 95.0 % → PUBLICAR


### 5.2 Reglas de calidad del mart (lo que se publica también se mide)

In [10]:
e = tablas["mart_entidades"]; p = tablas["mart_pares_entidad_proveedor"]
pesos = {f"R{i}": v["peso"] for i, v in enumerate(riesgo.cargar_reglas()["banderas"].values(), 1)}
recalc = sum(c[f"ind_r{i}_" + col.split(f"ind_r{i}_")[1]].astype(int) * pesos[f"R{i}"]
             for i in range(1, 10) for col in c.columns if col.startswith(f"ind_r{i}_")) * 100 / sum(pesos.values())
reglas_mart = [
    ("Completitud", "M-C1", "todo contrato tiene score y nivel", c[["num_score_riesgo", "niv_riesgo"]].notna().all(axis=1)),
    ("Validez", "M-V1", "score entre 0 y 100", c["num_score_riesgo"].between(0, 100)),
    ("Exactitud", "M-E1", "score = suma ponderada de banderas (recalculado)", (recalc.round(1) - c["num_score_riesgo"]).abs() < 0.11),
    ("Consistencia", "M-K1", "contratos del mart = filas de curated", pd.Series([len(c) == man_cur["filas_curated"]])),
    ("Consistencia", "M-K2", "suma de contratos por entidad = total", pd.Series([e["cant_contratos"].sum() == len(c)])),
    ("Unicidad", "M-U1", "id_contrato único en el mart", ~c["id_contrato"].duplicated(keep=False)),
    ("Oportunidad", "M-O1", "mart generado ≤ 24 h después del lote raw",
     pd.Series([(pd.Timestamp(man_mart["fin_utc"]) - pd.Timestamp(json.loads(riesgo.ultimo(RAW / "DS-01_*.manifest.json").read_text(encoding="utf-8"))["fin_utc"])) <= pd.Timedelta(hours=24)])),
]
tabla_mart = pd.DataFrame([{"dimension": d, "regla": r, "descripcion": t, "evaluados": len(s), "cumplen": int(s.sum()),
                            "cumplimiento_pct": round(100 * s.mean(), 2)} for d, r, t, s in reglas_mart])
tabla_mart.to_csv(GOB / "calidad_mart.csv", index=False, encoding="utf-8-sig")
tabla_mart

,dimension,regla,descripcion,evaluados,cumplen,cumplimiento_pct
0,Completitud,M-C1,todo contrato tiene score y nivel,294409,294409,100.0
1,Validez,M-V1,score entre 0 y 100,294409,294409,100.0
2,Exactitud,M-E1,score = suma ponderada de banderas (recalculado),294409,294409,100.0
3,Consistencia,M-K1,contratos del mart = filas de curated,1,1,100.0
4,Consistencia,M-K2,suma de contratos por entidad = total,1,1,100.0
5,Unicidad,M-U1,id_contrato único en el mart,294409,294409,100.0
6,Oportunidad,M-O1,mart generado ≤ 24 h después del lote raw,1,1,100.0


## 6. Linaje y trazabilidad: fuente → transformación → dataset → dashboard

La cadena se **verifica**: el SHA-256 que declara cada manifiesto se recalcula sobre el archivo. Si alguien altera un archivo intermedio, la cadena se rompe.

In [11]:
m01 = json.loads(riesgo.ultimo(RAW / "DS-01_*.manifest.json").read_text(encoding="utf-8"))
m02 = json.loads(riesgo.ultimo(RAW / "DS-02_*.manifest.json").read_text(encoding="utf-8"))
cur_file = CURATED / "DS-01_contratos_curated.parquet"
linaje = pd.DataFrame([
    {"paso": 1, "nodo": "API SODA jbjy-vk9h", "tipo": "fuente", "transformacion": f"SoQL: {m01['filtro_soql']}", "artefacto": m01["fuente"], "sha256_ok": None},
    {"paso": 2, "nodo": "raw DS-01", "tipo": "dataset", "transformacion": "descargar_fuentes.py (paginado, todo texto)", "artefacto": m01["archivo"],
     "sha256_ok": sha(RAW / m01["archivo"]) == m01["sha256"]},
    {"paso": 2, "nodo": "raw DS-02", "tipo": "dataset", "transformacion": "descargar_fuentes.py", "artefacto": m02["archivo"],
     "sha256_ok": sha(RAW / m02["archivo"]) == m02["sha256"]},
    {"paso": 3, "nodo": "curated DS-01", "tipo": "dataset", "transformacion": "limpieza.py: nulos, tipos, NIT, entidad, DIVIPOLA, calidad, HMAC, enmascarado",
     "artefacto": cur_file.name, "sha256_ok": sha(cur_file) == man_cur["sha256_curated"] == man_mart["entradas"]["curated"]["sha256"]},
    {"paso": 4, "nodo": "mart", "tipo": "dataset", "transformacion": f"riesgo.py + reglas v{man_mart['entradas']['reglas']['version']}",
     "artefacto": "mart_*.parquet", "sha256_ok": all(sha(MART / s["archivo"]) == s["sha256"] for s in man_mart["salidas"].values())},
    {"paso": 5, "nodo": "dashboard", "tipo": "consumo", "transformacion": "dashboard/app.py (lee solo mart, filtra por rol)", "artefacto": "streamlit run dashboard/app.py", "sha256_ok": None},
    {"paso": 5, "nodo": "alertas", "tipo": "consumo", "transformacion": "notificar_alertas.py → Power Automate", "artefacto": "automatizacion/alertas/", "sha256_ok": None},
])
assert man_cur["origen"].endswith(m01["archivo"]), "curated no proviene del último lote raw"
linaje.to_csv(GOB / "linaje.csv", index=False, encoding="utf-8-sig")
linaje

,paso,nodo,tipo,transformacion,artefacto,sha256_ok
0,1,API SODA jbjy-vk9h,fuente,SoQL: departamento='Antioquia' AND fecha_de_firma >= '2024-01-01',https://www.datos.gov.co/resource/jbjy-vk9h.json,None
1,2,raw DS-01,dataset,"descargar_fuentes.py (paginado, todo texto)",DS-01_secop2_contratos_20261105.parquet,True
2,2,raw DS-02,dataset,descargar_fuentes.py,DS-02_secop2_procesos_20261105.parquet,True
3,3,curated DS-01,dataset,"limpieza.py: nulos, tipos, NIT, entidad, DIVIPOLA, calidad, HMAC, enmascarado",DS-01_contratos_curated.parquet,True
4,4,mart,dataset,riesgo.py + reglas v1.0.0,mart_*.parquet,True
5,5,dashboard,consumo,"dashboard/app.py (lee solo mart, filtra por rol)",streamlit run dashboard/app.py,None
6,5,alertas,consumo,notificar_alertas.py → Power Automate,automatizacion/alertas/,None


Linaje a nivel de campo: cada columna del mart con su columna de origen y su transformación (extracto del diccionario).

In [12]:
diccionario[["columna", "origen", "transformacion"]].head(26)

,columna,origen,transformacion
0,id_contrato,DS-01.id_contrato,sin cambio
1,id_proceso_compra,DS-01.proceso_de_compra,sin cambio
2,nom_entidad,DS-01.nombre_entidad,quita * y / finales; nombre más frecuente por NIT
3,id_nit_entidad,DS-01.nit_entidad,deja 9 dígitos (retira el DV)
4,cod_orden_entidad,DS-01.orden,sin cambio
5,nom_municipio,DS-01.ciudad + DS-03.nom_mpio,"normaliza tildes, alias y cruza con DIVIPOLA"
6,cod_municipio,DS-03.cod_mpio,cruce por nombre normalizado
7,cod_modalidad,DS-01.modalidad_de_contratacion,nulos disfrazados -> nulo
8,cod_tipo_contrato,DS-01.tipo_de_contrato,nulos disfrazados -> nulo
9,cod_causal_modalidad,DS-01.justificacion_modalidad_de,nulos disfrazados -> nulo


## 7. Clasificación y sensibilidad (PII)

Cada columna de la fuente DS-01 se clasifica y se documenta **qué tratamiento recibió** en el pipeline. La clasificación se deriva de las mismas listas que ejecuta `limpieza.py`, así que documento y código no pueden divergir.

In [13]:
raw_cols = m01["columnas"]
cur_cols = set(pd.read_parquet(cur_file).columns)
seud = {v: k for k, v in limpieza.DOCUMENTOS.items()}
def clasificar(col):
    if col in seud:
        return "PII directa (documento)", "Confidencial", f"seudonimizada → {seud[col]} (HMAC-SHA256) y eliminada"
    if col in limpieza.NOMBRES_PERSONA or col == "proveedor_adjudicado":
        return "PII directa (nombre)", "Confidencial", "eliminada (proveedor jurídico se conserva como proveedor_nombre)"
    if col in ("nombre_del_banco", "tipo_de_cuenta", "n_mero_de_cuenta"):
        return "Dato financiero personal", "Confidencial", "eliminada: sin finalidad legítima"
    if "representante_legal" in col or col == "direcci_n_de_ejecuci_n_del_contrato" or col.startswith("tipo_de_documento"):
        return "PII indirecta", "Confidencial", "eliminada"
    if col in limpieza.TEXTO_LIBRE:
        return "Texto libre (puede contener PII)", "Interno", "documentos personales enmascarados como [DOC]"
    if col == "tipodocproveedor":
        return "Determina PII", "Interno", "resumida en proveedor_es_persona_natural"
    if col in cur_cols:
        return "No personal", "Público", "conservada"
    return "No personal", "Público", "eliminada por vacía (≥ 99 % nulos) o sin uso"
clasif = pd.DataFrame([(c_, *clasificar(c_)) for c_ in raw_cols], columns=["columna_fuente", "tipo_dato", "clasificacion_raw", "tratamiento"])
clasif.to_csv(GOB / "clasificacion_campos_DS-01.csv", index=False, encoding="utf-8-sig")
display(clasif.groupby(["tipo_dato", "clasificacion_raw"]).size().rename("columnas").to_frame())
clasif[clasif["tipo_dato"] != "No personal"]

,,columnas
tipo_dato,clasificacion_raw,
Dato financiero personal,Confidencial,3
Determina PII,Interno,1
No personal,Público,69
PII directa (documento),Confidencial,5
PII directa (nombre),Confidencial,5
PII indirecta,Confidencial,8
Texto libre (puede contener PII),Interno,4


,columna_fuente,tipo_dato,clasificacion_raw,tratamiento
14,descripcion_del_proceso,Texto libre (puede contener PII),Interno,documentos personales enmascarados como [DOC]
17,justificacion_modalidad_de,Texto libre (puede contener PII),Interno,documentos personales enmascarados como [DOC]
21,condiciones_de_entrega,Texto libre (puede contener PII),Interno,documentos personales enmascarados como [DOC]
22,tipodocproveedor,Determina PII,Interno,resumida en proveedor_es_persona_natural
23,documento_proveedor,PII directa (documento),Confidencial,seudonimizada → proveedor_id (HMAC-SHA256) y eliminada
24,proveedor_adjudicado,PII directa (nombre),Confidencial,eliminada (proveedor jurídico se conserva como proveedor_nombre)
49,nombre_representante_legal,PII directa (nombre),Confidencial,eliminada (proveedor jurídico se conserva como proveedor_nombre)
50,nacionalidad_representante_legal,PII indirecta,Confidencial,eliminada
51,domicilio_representante_legal,PII indirecta,Confidencial,eliminada
52,tipo_de_identificaci_n_representante_legal,PII indirecta,Confidencial,eliminada


In [14]:
print("Clasificación por capa:")
display(pd.DataFrame(gobierno.CLASIFICACION, index=["nivel", "motivo"]).T)
print("Ley 1581 · datos sensibles (art. 5: salud, religión, biometría, orientación…): 0 columnas en DS-01")
print("PII en claro en curated:", man_cur["pii_en_claro_en_curated"], "· en mart:", man_mart["controles_gobierno"]["pii_en_claro_mart"])

Clasificación por capa:


,nivel,motivo
raw,Confidencial,"Contiene PII en claro (cédulas, nombres, cuentas bancarias). Evidencia inmut..."
curated,Interno,PII seudonimizada con HMAC-SHA256; texto libre enmascarado; sin columnas ban...
mart,Confidencial,"Sin PII en claro, pero las alertas de riesgo son reservadas hasta que exista..."
dashboard_publico,Público,"Solo agregados (por mes, modalidad y municipio), sin alertas individuales."


Ley 1581 · datos sensibles (art. 5: salud, religión, biometría, orientación…): 0 columnas en DS-01
PII en claro en curated: 0 · en mart: 0


## 8. Control de acceso

La matriz no es solo una tabla: `gobierno.ruta_capa(rol, capa)` es la **única** forma en que el dashboard obtiene la ruta de una capa, y lanza `AccesoDenegado` si el rol no tiene permiso. Aquí se prueba para todas las combinaciones.

In [15]:
acc = pd.DataFrame(gobierno.MATRIZ_ACCESO).T
acc["justificacion"] = pd.Series(gobierno.JUSTIFICACION_ACCESO)
acc.to_csv(GOB / "matriz_acceso.csv", encoding="utf-8-sig")
display(acc)
prueba = []
for rol in gobierno.ROLES:
    for capa in gobierno.CLASIFICACION:
        for accion in ("L", "E"):
            try:
                gobierno.ruta_capa(rol, capa, accion); r = "permitido"
            except gobierno.AccesoDenegado:
                r = "denegado"
            prueba.append({"rol": rol, "capa": capa, "accion": accion, "resultado": r})
prueba = pd.DataFrame(prueba).pivot_table(index="rol", columns=["capa", "accion"], values="resultado", aggfunc="first")
escritores = [r for r in gobierno.ROLES if gobierno.puede(r, "mart", "E")]
assert escritores == ["custodio"], "Segregación: solo el custodio escribe"
print("Segregación de funciones OK: solo", escritores, "escribe en las capas")
prueba

,raw,curated,mart,dashboard_publico,justificacion
data_owner,L,L,L,L,Responde por el dato y aprueba accesos; lee raw solo para resolver disputas ...
data_steward,L,L,L,L,Mide la calidad y resuelve incidentes: necesita ver el dato original para di...
custodio,E,E,E,E,Opera el pipeline: es el único rol que escribe en las capas (segregación de ...
analista,-,L,L,L,Construye banderas y dashboard sobre datos seudonimizados; nunca necesita la...
auditor,-,-,L,L,Usa las alertas priorizadas; la evidencia la consulta en SECOP con el enlace...
ciudadania,-,-,-,L,Transparencia: ve agregados; las alertas individuales son reservadas (no son...


Segregación de funciones OK: solo ['custodio'] escribe en las capas


capa            curated            dashboard_publico                  mart                   raw           
accion                E          L                 E          L          E          L          E          L
rol                                                                                                        
analista       denegado  permitido          denegado  permitido   denegado  permitido   denegado   denegado
auditor        denegado   denegado          denegado  permitido   denegado  permitido   denegado   denegado
ciudadania     denegado   denegado          denegado  permitido   denegado   denegado   denegado   denegado
custodio      permitido  permitido         permitido  permitido  permitido  permitido  permitido  permitido
data_owner     denegado  permitido          denegado  permitido   denegado  permitido   denegado  permitido
data_steward   denegado  permitido          denegado  permitido   denegado  permitido   denegado  permitido

In [16]:
print("Pestañas del dashboard por rol (derivadas de la matriz):")
pd.DataFrame({r: {v: "✔" if r in roles else "" for v, roles in gobierno.VISTAS_DASHBOARD.items()} for r in gobierno.ROLES})

Pestañas del dashboard por rol (derivadas de la matriz):


,data_owner,data_steward,custodio,analista,auditor,ciudadania
Resumen,✔,✔,✔,✔,✔,✔
P1 · Entidades,✔,✔,,✔,✔,
P2 · Ventanas atípicas,✔,✔,✔,✔,✔,✔
P3 · Entidad–proveedor,✔,✔,,✔,✔,
P4–P6 · Contratos,✔,✔,,✔,✔,
Bandeja de alertas,✔,,,✔,✔,
Gobierno y calidad,✔,✔,✔,✔,,


## 9. KPI de gobierno y madurez al cierre

In [17]:
reg = ROOT / "automatizacion" / "registro_pipeline.csv"
kpi = pd.DataFrame([
    {"KPI": "Calidad por lote", "meta": "≥ 95 %", "resultado": f"{man_cur['calidad']['puntaje_lote']} %"},
    {"KPI": "Protección de PII (campos en claro en consumo)", "meta": "0", "resultado": man_mart["controles_gobierno"]["pii_en_claro_mart"]},
    {"KPI": "Cobertura de riesgo (contratos con score)", "meta": "100 %", "resultado": f"{100 * c['num_score_riesgo'].notna().mean():.1f} %"},
    {"KPI": "Activos con Owner y Steward", "meta": "100 %", "resultado": f"{100 * catalogo[['Owner', 'Steward']].notna().all(axis=1).mean():.0f} %"},
    {"KPI": "Columnas del mart en el diccionario", "meta": "100 %", "resultado": f"{100 * (1 - len(faltan) / c.shape[1]):.0f} %"},
    {"KPI": "Columnas del mart en el estándar de nombres", "meta": "100 %", "resultado": f"{100 * (1 - sum(len(x) for x in nom['no_conformes']) / nom['columnas'].sum()):.0f} %"},
    {"KPI": "Eslabones de linaje verificados por SHA-256", "meta": "100 %", "resultado": f"{100 * linaje['sha256_ok'].dropna().mean():.0f} %"},
    {"KPI": "Frescura (horas lote raw → mart)", "meta": "≤ 24 h",
     "resultado": round((pd.Timestamp(man_mart['fin_utc']) - pd.Timestamp(m01['fin_utc'])).total_seconds() / 3600, 1)},
    {"KPI": "Corridas automáticas registradas", "meta": "≥ 1", "resultado": len(pd.read_csv(reg)) if reg.exists() else 0},
])
kpi.to_csv(GOB / "kpi_gobierno.csv", index=False, encoding="utf-8-sig")
kpi

,KPI,meta,resultado
0,Calidad por lote,≥ 95 %,99.66 %
1,Protección de PII (campos en claro en consumo),0,0
2,Cobertura de riesgo (contratos con score),100 %,100.0 %
3,Activos con Owner y Steward,100 %,100 %
4,Columnas del mart en el diccionario,100 %,100 %
5,Columnas del mart en el estándar de nombres,100 %,100 %
6,Eslabones de linaje verificados por SHA-256,100 %,100 %
7,Frescura (horas lote raw → mart),≤ 24 h,1.1
8,Corridas automáticas registradas,≥ 1,2


**Madurez (instrumento de la Sesión 1, /16):** AS-IS 2 → Entrega 2: 9 → **cierre: 13 (Nivel 4 · Gestionado)**.
Las afirmaciones 2, 4, 5, 7 y 8 pasan a *Sí* con la evidencia de este notebook (catálogo automático, calidad medida por lote con gate,
clasificación por campo aplicada, linaje verificado, políticas escritas y ejecutadas). Las afirmaciones 1, 3 y 6 quedan en *Parcial*
por las razones declaradas desde la Entrega 2: el Owner es simulado, el diccionario documenta los campos que se usan y el control
de acceso se implementa en la aplicación, no con IAM institucional. Detalle en el Plan de Gobierno §10.